# Session 2: the membrane is an RC circuit
**A. Real-world problem.** A cell membrane keeps charge apart, so it stores energy like a
capacitor C; ion channels let charge leak through, like a resistor R. Inject a current I
and the voltage obeys  C dV/dt = -(V - V_rest)/R + I.  Add a rule "when V reaches the
threshold, record a spike, reset V, and wait a refractory time" and you have the *leaky
integrate-and-fire* (LIF) neuron. How fast does it fire for a given current, and what current
do we need to make it fire 20 times a second?

*General education only. All numbers labelled typical are textbook-style round values,
not measurements of a particular cell.*

In [ ]:
import math
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

V_REST = -70.0     # mV, typical
V_TH = -55.0       # mV, typical threshold
V_RESET = -70.0    # mV, typical reset value
R_DEFAULT = 100.0  # megaohm (typical order of magnitude for a small neuron)
C_DEFAULT = 100.0  # picofarad  -> tau = R C = 10 ms (typical)
T_REF = 2.0        # ms, typical refractory period

## Step 1: units and the time constant
Units: 1 megaohm x 1 nanoamp = 1 mV, and 1 megaohm x 1 picofarad = 1 microsecond, so
tau (ms) = R (megaohm) x C (pF) / 1000. With R = 100 and C = 100 this gives 10 ms.

For a constant current the equation has the exact solution

    V(t) = V_rest + R I (1 - exp(-t / tau))

which climbs towards the level V_rest + R I. It can only reach the threshold if
R I > V_th - V_rest (15 mV here): the *rheobase* current is I = (V_th - V_rest) / R.

In [ ]:
def tau_ms(R=R_DEFAULT, C=C_DEFAULT):
    return R * C / 1000.0


def rheobase_nA(R=R_DEFAULT):
    """Smallest constant current (nA) that can ever reach the threshold."""
    return (V_TH - V_REST) / R


def v_subthreshold(t, I_nA, R=R_DEFAULT, C=C_DEFAULT):
    """Exact V(t) for a constant current switched on at t = 0 (valid until the threshold)."""
    return V_REST + R * I_nA * (1.0 - np.exp(-np.asarray(t, dtype=float) / tau_ms(R, C)))

## Step 2: the firing rate, derived
Solve V(T) = V_th for the time T to climb from reset to threshold. With the reset equal to
the rest value:

    V_rest + R I (1 - exp(-T/tau)) = V_th   =>   exp(-T/tau) = 1 - (V_th - V_rest)/(R I)
    T = tau ln( R I / (R I - (V_th - V_rest)) )

After a spike the cell is also silent for t_ref, so one full cycle lasts T + t_ref and
the rate is  f = 1 / (t_ref + tau ln(R I / (R I - (V_th - V_rest)))).
With t_ref = 0 this is the closed form  f = 1 / (tau ln(...)).  Below the rheobase f = 0.

In [ ]:
def climb_time_ms(I_nA, R=R_DEFAULT, C=C_DEFAULT):
    """Time (ms) to climb from reset to threshold; inf if the current is too small."""
    ri = R * I_nA
    dv = V_TH - V_RESET
    if ri <= dv:
        return math.inf
    return tau_ms(R, C) * math.log(ri / (ri - dv))


def rate_hz(I_nA, R=R_DEFAULT, C=C_DEFAULT, t_ref=T_REF):
    """Closed-form firing rate (spikes per second)."""
    T = climb_time_ms(I_nA, R, C)
    return 0.0 if math.isinf(T) else 1000.0 / (t_ref + T)


def current_for_rate(f_hz, R=R_DEFAULT, C=C_DEFAULT, t_ref=T_REF):
    """Design problem, inverted: the constant current (nA) that gives f_hz spikes per second.
    From T = 1000/f - t_ref:  R I = (V_th - V_reset) / (1 - exp(-T/tau))."""
    T = 1000.0 / f_hz - t_ref
    if T <= 0:
        return math.inf
    return (V_TH - V_RESET) / (R * (1.0 - math.exp(-T / tau_ms(R, C))))

## Step 3: simulate it numerically
Between spikes we can step the equation exactly (constant current in a step), or with
the simple Euler rule  V += dt * (-(V - V_rest)/tau + R I / tau).  Comparing them with the
closed form shows how good each is.

In [ ]:
def simulate_lif(I_nA, t_end=500.0, dt=0.01, R=R_DEFAULT, C=C_DEFAULT, t_ref=T_REF, method="exact"):
    """Returns (t, V, spike_times). I_nA may be a number or a function of time (ms)."""
    tau = tau_ms(R, C)
    cur = I_nA if callable(I_nA) else (lambda t, a=I_nA: a)
    n = int(round(t_end / dt))
    t = np.arange(n + 1) * dt
    V = np.empty(n + 1)
    V[0] = V_REST
    spikes = []
    wait = 0.0
    a = 1.0 - math.exp(-dt / tau)
    for i in range(n):
        v = V[i]
        if wait > 1e-12:
            wait -= dt
            V[i + 1] = V_RESET
            continue
        vinf = V_REST + R * cur(t[i])
        if method == "exact":
            v = v + (vinf - v) * a
        else:
            v = v + dt * (vinf - v) / tau
        if v >= V_TH:
            spikes.append(t[i + 1])
            v = V_RESET
            wait = t_ref
        V[i + 1] = v
    return t, V, np.array(spikes)


def measured_rate(spikes):
    """Rate from the last inter-spike interval (spikes per second)."""
    if len(spikes) < 2:
        return 0.0
    return 1000.0 / float(spikes[-1] - spikes[-2])

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    print(f"tau = {tau_ms():.1f} ms, rheobase = {rheobase_nA():.3f} nA")
    I20 = current_for_rate(20.0)
    print(f"current for 20 spikes per second (t_ref {T_REF} ms): {I20:.4f} nA")
    for I in (0.15, 0.16, 0.2, 0.3, 0.5, 1.0):
        t, V, sp = simulate_lif(I, 1000.0, 0.01)
        print(f"I = {I:5.2f} nA: closed form {rate_hz(I):7.2f} Hz   simulation {measured_rate(sp):7.2f} Hz   "
              f"(without refractory {rate_hz(I, t_ref=0.0):7.2f} Hz)")

    fig, ax = plt.subplots(2, 1, figsize=(7, 6))
    for I, col in ((0.12, "gray"), (0.2, "#1f6feb"), (0.4, "#d9534f")):
        t, V, sp = simulate_lif(I, 150.0, 0.01)
        trace = V.copy()
        for s in sp:
            trace[int(round(s / 0.01))] = 20.0
        ax[0].plot(t, trace, color=col, label=f"I = {I} nA")
    ax[0].axhline(V_TH, color="k", ls="--", lw=1)
    ax[0].set_xlabel("time (ms)")
    ax[0].set_ylabel("membrane voltage (mV)")
    ax[0].set_title("Leaky integrate-and-fire neuron (spikes drawn to +20 mV)")
    ax[0].legend(fontsize=8)
    ax[0].grid(alpha=0.3)
    cur = np.linspace(0.0, 1.0, 400)
    ax[1].plot(cur, [rate_hz(c) for c in cur], label="closed form, t_ref 2 ms")
    ax[1].plot(cur, [rate_hz(c, t_ref=0.0) for c in cur], "--", label="closed form, t_ref 0")
    cs = np.linspace(0.05, 1.0, 12)
    ax[1].plot(cs, [measured_rate(simulate_lif(c, 1000.0, 0.01)[2]) for c in cs], "ko", ms=4, label="simulation")
    ax[1].axvline(rheobase_nA(), color="gray", ls=":")
    ax[1].set_xlabel("input current (nA)")
    ax[1].set_ylabel("firing rate (spikes per second)")
    ax[1].set_title("The f-I curve")
    ax[1].legend(fontsize=8)
    ax[1].grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "lif_and_fi.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6, 4))
    t = np.linspace(0, 60, 300)
    ax2.plot(t, v_subthreshold(t, 0.1), label="exact V(t), I = 0.1 nA")
    for dt, st in ((5.0, "s"), (1.0, "o")):
        tt, VV, _ = simulate_lif(0.1, 60.0, dt, method="euler")
        ax2.plot(tt, VV, st, ms=3, label=f"Euler, dt = {dt} ms")
    ax2.set_xlabel("time (ms)")
    ax2.set_ylabel("membrane voltage (mV)")
    ax2.set_title("Subthreshold response: exact versus Euler")
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "subthreshold.png", dpi=120)

## Coding tasks
1. **Easy.** Use `rate_hz` to print the rate for I = 0.151, 0.152, 0.16 and 0.2 nA, and predict first which one will be the largest relative jump.
2. **Stretch.** Write `time_to_rate(f)` that returns the current needed for f spikes per second for C = 200 pF and R = 100 megaohm (use `current_for_rate`), and compare it with the C = 100 pF result.
3. **Explain.** Explain why the numerical rate from the simulation is a tiny bit lower than the closed form (63.01 against 63.04 Hz for 0.2 nA). Hint: when is the threshold checked?

## Assessment
- **A. Conceptual.** If R stays the same but the capacitance C is halved, does the neuron fire faster or slower for the same current? Explain without equations, then check with a number.
- **B. Numerical.** For R = 100 megaohm, C = 100 pF and I = 0.3 nA find the climb time T and the rate f (t_ref = 2 ms).
- **C. Graph and simulation interpretation.** Describe the shape of the f-I curve: what happens below 0.15 nA, just above it, and for very large currents? What does the limit mean?
- **D. Design and optimisation challenge.** Design: the cell must fire at 20 spikes per second, within 18 to 22. Find the current for exactly 20 per second, the allowed range of current, and say what makes this design delicate.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("tau = R C = 100 megaohm x 100 pF = 10 ms", abs(L.tau_ms() - 10.0) < 1e-12))
    checks.append(("rheobase = 15 mV / 100 megaohm = 0.15 nA", abs(L.rheobase_nA() - 0.15) < 1e-12))
    v10 = float(L.v_subthreshold(10.0, 0.1))
    checks.append(("after one tau the voltage has covered 63.2 percent of its climb (-63.679 mV for 0.1 nA)",
                   abs(v10 - (-70 + 10 * (1 - math.exp(-1)))) < 1e-9 and abs(v10 + 63.679) < 1e-3))
    checks.append(("V(0) is the resting value", abs(float(L.v_subthreshold(0.0, 0.1)) - L.V_REST) < 1e-12))
    # the closed form really solves V(T) = V_th
    for I in (0.16, 0.2, 0.5):
        T = L.climb_time_ms(I)
        checks.append((f"derivation: V(T) equals the threshold at I = {I} nA", abs(float(L.v_subthreshold(T, I)) - L.V_TH) < 1e-9))
    checks.append(("worked example: I = 0.2 nA gives T = 10 ln 4 = 13.863 ms", abs(L.climb_time_ms(0.2) - 10 * math.log(4)) < 1e-9 and abs(L.climb_time_ms(0.2) - 13.863) < 1e-3))
    checks.append(("worked example: with t_ref = 2 ms the closed-form rate at 0.2 nA is 63.04 Hz", abs(L.rate_hz(0.2) - 63.04) < 0.01))
    checks.append(("worked example: without refractory period it is 72.13 Hz", abs(L.rate_hz(0.2, t_ref=0.0) - 72.13) < 0.01))
    t, V, sp = L.simulate_lif(0.2, 1000.0, 0.01)
    checks.append((f"validation: simulation gives {L.measured_rate(sp):.2f} Hz, within 0.1 Hz of the closed form", abs(L.measured_rate(sp) - L.rate_hz(0.2)) < 0.1))
    checks.append(("the first spike of the 0.2 nA run is at 13.87 ms", abs(sp[0] - 13.87) < 0.011))
    checks.append(("every inter-spike interval of the 0.2 nA run is 15.87 ms", bool(np.all(np.abs(np.diff(sp) - 15.87) < 0.011))))
    # below / at the rheobase
    checks.append(("no firing below the rheobase (closed form and simulation, 0.149 nA)",
                   L.rate_hz(0.149) == 0.0 and len(L.simulate_lif(0.149, 500.0, 0.01)[2]) == 0))
    checks.append(("edge case: exactly at the rheobase (0.15 nA) the neuron never fires", L.rate_hz(0.15) == 0.0 and len(L.simulate_lif(0.15, 500.0, 0.01)[2]) == 0))
    checks.append(("just above the rheobase the rate jumps: 0.16 nA gives 33.64 Hz", abs(L.rate_hz(0.16) - 33.64) < 0.01))
    f = [L.rate_hz(c) for c in np.linspace(0.151, 2.0, 200)]
    checks.append(("the f-I curve rises with current", bool(np.all(np.diff(f) > 0))))
    checks.append(("a huge current saturates near 1 / t_ref = 500 Hz", abs(L.rate_hz(1e3) - 500.0) < 0.5 and L.rate_hz(1e6) < 500.0))
    # design problem
    I20 = L.current_for_rate(20.0)
    checks.append(("worked example: 20 spikes per second needs I = 0.1512 nA", abs(I20 - 0.1512) < 1e-4))
    checks.append(("design check: that current gives 20.00 Hz in the closed form and 20 Hz (within 0.1) in simulation",
                   abs(L.rate_hz(I20) - 20.0) < 1e-9 and abs(L.measured_rate(L.simulate_lif(I20, 2000.0, 0.01)[2]) - 20.0) < 0.1))
    checks.append(("design sensitivity: 0.151 nA gives 19.17 Hz and 0.152 nA gives 22.07 Hz", abs(L.rate_hz(0.151) - 19.17) < 0.01 and abs(L.rate_hz(0.152) - 22.07) < 0.01))
    checks.append(("scaling: doubling R halves the current needed (R = 200 megaohm, C = 50 pF: 0.0756 nA)",
                   abs(L.current_for_rate(20.0, R=200.0, C=50.0) - I20 / 2) < 1e-9 and abs(L.current_for_rate(20.0, R=200.0, C=50.0) - 0.0756) < 1e-4))
    checks.append(("a rate above 1 / t_ref cannot be reached", math.isinf(L.current_for_rate(600.0)) and L.current_for_rate(100.0) > 0))
    # Euler versus exact
    e1 = np.max(np.abs(L.simulate_lif(0.1, 60.0, 1.0, method="euler")[1] - L.v_subthreshold(np.arange(61) * 1.0, 0.1)))
    e2 = np.max(np.abs(L.simulate_lif(0.1, 60.0, 0.1, method="euler")[1] - L.v_subthreshold(np.arange(601) * 0.1, 0.1)))
    checks.append((f"Euler error shrinks about 10 times when dt shrinks 10 times ({e1:.3f} mV, {e2:.4f} mV)", 8 < e1 / e2 < 12))
    ex = np.max(np.abs(L.simulate_lif(0.1, 60.0, 5.0, method="exact")[1] - L.v_subthreshold(np.arange(13) * 5.0, 0.1)))
    checks.append(("the exact step has no time-step error even for dt = 5 ms", ex < 1e-9))
    checks.append(("question B: at 0.3 nA T = 10 ln 2 = 6.931 ms and the rate is 111.96 Hz", abs(L.climb_time_ms(0.3) - 10 * math.log(2)) < 1e-9 and abs(L.rate_hz(0.3) - 111.96) < 0.01))
    checks.append(("question A: halving C (50 pF) raises the 0.3 nA rate to 182.96 Hz; doubling C (200 pF) lowers it to 63.04 Hz",
                   abs(L.rate_hz(0.3, C=50.0) - 182.96) < 0.01 and abs(L.rate_hz(0.3, C=200.0) - 63.04) < 0.01))
    checks.append(("question D: 18 spikes per second needs 0.1507 nA and 22 needs 0.1520 nA", abs(L.current_for_rate(18.0) - 0.1507) < 1e-4 and abs(L.current_for_rate(22.0) - 0.1520) < 1e-4))
    checks.append(("bench experiment: R = 10 kohm and C = 1 uF give tau = 10 ms", abs(L.tau_ms(0.01, 1e6) - 10.0) < 1e-9))
    checks.append(("exercise 3: the simulated rate at 0.2 nA is 63.01 Hz against 63.04 Hz in the closed form (the threshold is checked on a time grid)", abs(L.measured_rate(sp) - 63.01) < 0.01 and abs(L.rate_hz(0.2) - 63.04) < 0.01))
    checks.append(("question D: for 20 per second T = 48 ms and R I = 15 / (1 - exp(-4.8)) = 15.12 mV", abs(1000 / 20.0 - L.T_REF - 48.0) < 1e-12 and abs(15 / (1 - math.exp(-4.8)) - 15.12) < 0.005))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
